# See Water through the Years and Seasons

Compare annual and seasonal water values, fortnightly vegetation and water, and groundwater context.

Run the cells in order. Change the place, identifier or columns to explore other records. Downloads from GeoLibre use your selected tehsil; these templates start with Hilsa, Nalanda, Bihar.


## Set up Python

Run the collapsed setup cells. They import the libraries and define `read_json`, a small response reader. It reads JSON text, treats non-standard `NaN` and `Infinity` numbers as missing, and also accepts JSON returned inside a string. HTTP errors and malformed responses remain visible. Expand the cells to read the code.


In [ ]:
import sys
if sys.platform == "emscripten":
    import micropip
    await micropip.install(["geopandas", "matplotlib", "requests", "pyodide-http"])
    import pyodide_http
    pyodide_http.patch_all()

import os
import re
import ast
import json
from getpass import getpass
from inspect import isawaitable
from urllib.parse import urljoin
import requests
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, FileLink
plt.rcParams.update({"axes.spines.top": False, "axes.spines.right": False})


In [ ]:
SCOPE = json.loads("{\"state\": \"Bihar\", \"district\": \"Nalanda\", \"tehsil\": \"Hilsa\"}")
API_URL = 'https://geoserver.core-stack.org/api/v1/'
STAC_URL = 'https://spatio-temporal-asset-catalog.s3.ap-south-1.amazonaws.com/CorestackCatalogs_merged_collection/tehsil_wise/catalog.json'
YEARS = list(range(2017, 2025))


In [ ]:
"""Small response reader embedded in the notebooks' collapsed setup cell."""
import json


def read_json(response):
    """Read JSON text; represent non-standard NaN/Infinity values as missing."""
    response.raise_for_status()
    raw_text = response.text.lstrip("\ufeff")
    try:
        # Some API tables contain bare NaN or Infinity, which are not JSON numbers.
        data = json.loads(raw_text, parse_constant=lambda value: None)
        # Also accept a JSON document returned as a JSON-encoded string.
        if isinstance(data, str):
            data = json.loads(data.lstrip("\ufeff"), parse_constant=lambda value: None)
        return data
    except ValueError as error:
        raise ValueError(
            "The server response is not readable JSON. "
            "Inspect response.status_code and response.text[:500], then retry the request."
        ) from error


## Choose the place and set your API key

Edit `SCOPE` in the setup cell to change the place. The [public API guide](https://docs.core-stack.org/use-precomputed-data/public-apis/) explains registration and API keys. This cell reuses `CORE_STACK_API_KEY` or asks for it privately, then stores it in this kernel’s environment. The key is sent only to the API, in the `X-API-Key` header. Restart the kernel and run from the top after changing places.


In [ ]:
place = {key: re.sub(r"[\s_]+", "_", SCOPE[key].replace("(", "").replace(")", "")).strip("_").lower()
         for key in ["state", "district", "tehsil"]}
state, district, tehsil = place["state"], place["district"], place["tehsil"]
api_key = os.environ.get("CORE_STACK_API_KEY", "").strip()
if not api_key:
    api_key = getpass("CoRE Stack API key: ")
    if isawaitable(api_key):
        api_key = await api_key
os.environ["CORE_STACK_API_KEY"] = str(api_key).strip()
api_headers = {"X-API-Key": os.environ["CORE_STACK_API_KEY"]}
display(place)


## Read the tehsil and choose a micro-watershed

One request returns the tehsil’s tables. The cell keeps the tables used here, lists identifiers and shows the first record’s first ten fields. Change the selected identifier, then rerun the following cells. Blank fields mean the source did not supply a value.


In [ ]:
response = requests.get(API_URL + "get_tehsil_data/", params=place, headers=api_headers, timeout=180)
api_data = read_json(response)
required_tables = ['hydrological_annual', 'hydrological_seasonal', 'soge_vector', 'aquifer_vector']
tables = {name: pd.DataFrame(api_data.get(name, [])) for name in required_tables}
display(pd.DataFrame({"Table": required_tables, "Rows": [len(tables[name]) for name in required_tables]}))
mws_table = tables['hydrological_annual']
display(mws_table[["uid"]])
mws_id = str(mws_table.iloc[0]["uid"])  # Choose another ID from the list.
selected = mws_table.loc[mws_table["uid"].astype(str) == mws_id].iloc[0]
display(selected.iloc[:10].to_frame("First 10 fields"))


## Discover data and descriptions in STAC

STAC lists published datasets, field descriptions, downloads and styles. Change `dataset` to another item from the collection. Asset links are used as published, wherever the files are hosted. STAC describes asset fields; API tables may use different names and units, which are shown explicitly in the examples below.


In [ ]:
collection_url = urljoin(STAC_URL, f"{state}/{district}/{tehsil}/collection.json")
response = requests.get(collection_url, timeout=90)
collection = read_json(response)
items = pd.DataFrame([{"Item": link["href"].split("/")[-1].removesuffix(".json"),
                       "URL": urljoin(collection_url, link["href"])}
                      for link in collection["links"] if link["rel"] == "item"], columns=["Item", "URL"])
# Follow a relevant item link from the collection.
dataset = "water_balance_fortnightly_vector"
matches = items.loc[items["Item"].str.endswith("_" + dataset)]
item = None
if not matches.empty:
    item_url = matches.iloc[0]["URL"]
    response = requests.get(item_url, timeout=90)
    item = read_json(response)
    display(Markdown(item["properties"].get("description", "No description published.")))
    field_notes = pd.DataFrame(item["properties"].get("table:columns", []))
    display(field_notes.reindex(columns=["name", "type", "description"]).head(12))
    print("Published field count:", len(field_notes), "— use field_notes to see them all.")
    display(pd.DataFrame(item["assets"]).T.reindex(columns=["title", "type", "href"]))
else:
    print("This dataset is not listed in the tehsil's STAC collection. Available items:")
    display(items)


## Annual and seasonal water together

Each column shows one water measure. Annual values sit above Kharif, Rabi and Zaid, with the same scale within each column. All water values are millimetres; years run July–June. Missing years remain gaps. These are separately published annual and seasonal summaries; this cell does not calculate one from the other.


In [ ]:
annual = selected
seasonal = tables["hydrological_seasonal"].set_index("uid").reindex([mws_id]).iloc[0]
measures = {"precipitation": "Rainfall", "et": "Evapotranspiration", "runoff": "Runoff"}
periods = ["Annual", "Kharif", "Rabi", "Zaid"]
water = pd.DataFrame([{ "Year": year, "Period": period, "Measure": label,
                       "Water (mm)": (annual if period == "Annual" else seasonal).get(
                           f"{field}{'' if period == 'Annual' else '_' + period.lower()}_in_mm_{year}-{year + 1}")}
                      for year in YEARS for period in periods for field, label in measures.items()])
water["Water (mm)"] = pd.to_numeric(water["Water (mm)"], errors="coerce")
display(water.pivot(index="Year", columns=["Period", "Measure"], values="Water (mm)"))
fig, axes = plt.subplots(4, 3, figsize=(12, 9), sharex=True, sharey="col")
for row, period in enumerate(periods):
    for col, label in enumerate(measures.values()):
        values = water.loc[(water["Period"] == period) & (water["Measure"] == label)]
        axes[row, col].plot(values["Year"], values["Water (mm)"], marker="o", color=["#247ba0", "#df9c32", "#528a64"][col])
        axes[row, col].set_title(f"{period} · {label}")
        axes[row, col].set_ylim(bottom=0)
        axes[row, col].grid(alpha=0.2)
    axes[row, 0].set_ylabel("mm")
for ax in axes[-1]:
    ax.set_xticks(YEARS, [f"{y}–{str(y+1)[-2:]}" for y in YEARS], rotation=45)
fig.suptitle(f"Water through the years · {mws_id}")
plt.tight_layout()
plt.show()


## Fortnightly water and vegetation

The MWS API returns water and NDVI with their actual dates. NDVI is unitless, so it has its own axis. Change the date slice to look closely at a season.


In [ ]:
response = requests.get(API_URL + "get_mws_data/", params={**place, "mws_id": mws_id}, headers=api_headers, timeout=180)
series = pd.DataFrame(read_json(response)["time_series"])
series["date"] = pd.to_datetime(series["date"])
series = series.set_index("date").sort_index()
columns = ["precipitation", "et", "runoff", "ndvi_crop", "ndvi_tree", "ndvi_shrub"]
series = series.reindex(columns=columns).apply(pd.to_numeric, errors="coerce")
view = series.loc["2017-07-01":"2025-06-30"]
display(view.head())
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
view[["precipitation", "et", "runoff"]].rename(columns={"precipitation": "Rainfall", "et": "ET", "runoff": "Runoff"}).plot(ax=axes[0], ylabel="Water (mm)")
view[["ndvi_crop", "ndvi_tree", "ndvi_shrub"]].rename(columns={"ndvi_crop": "Crops", "ndvi_tree": "Trees", "ndvi_shrub": "Shrubs"}).plot(ax=axes[1], ylabel="NDVI (unitless)")
axes[1].set_ylim(-1, 1)
axes[0].set_title(f"Fortnightly water and vegetation · {mws_id}")
plt.tight_layout()
plt.show()


## Groundwater and aquifers

Well depth and change in groundwater storage use separate units. The extraction class and aquifer composition provide context; they are not annual measurements.


In [ ]:
extraction = tables["soge_vector"].set_index("uid").reindex([mws_id]).iloc[0]
aquifer = tables["aquifer_vector"].set_index("uid").reindex([mws_id]).iloc[0]
display(extraction.reindex(["soge_dev_percent", "class_name"]).to_frame("Groundwater extraction"))
display(aquifer.to_frame("Aquifer record"))
groundwater = pd.DataFrame({"Well depth (m)": [annual.get(f"welldepth_in_m_{y}-{y+1}") for y in YEARS],
                            "Change in storage (mm)": [annual.get(f"deltag_in_mm_{y}-{y+1}") for y in YEARS]}, index=YEARS).apply(pd.to_numeric, errors="coerce")
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, column in zip(axes, groundwater):
    groundwater[column].plot(ax=ax, marker="o", title=column, ylabel=column)
    ax.axhline(0, color="gray", linewidth=0.6)
plt.tight_layout()
plt.show()
